In [13]:
!pip install -q pyspark
from google.colab import drive
drive.mount('/content/drive')

from pyspark.sql import SparkSession
spark = (SparkSession.builder
         .appName("YelpSentiment")
         .config("spark.driver.memory", "10g")
         .getOrCreate())

df       = spark.read.parquet("/content/drive/MyDrive/yelp_project/reviews_clean.parquet")
business = spark.read.parquet("/content/drive/MyDrive/yelp_project/business.parquet")

# while developing: work on a 5% sample, then run the final version on all data
sample = df.sample(fraction=0.05, seed=42)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [14]:

import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import os

from pyspark.sql.functions import (
    col, length, size, split, avg, expr
)

# Create a folder for saving charts
os.makedirs("figures", exist_ok=True)

# Set chart style
sns.set_theme(style="whitegrid")



# Exploratory Data Analysis (EDA)

This section explores the Yelp reviews dataset to understand its distribution, review lengths, and changes in review activity over time.

The analysis includes star and sentiment distribution, review length comparison, and yearly review trends.



### Task 1 — Star and Label Distribution

This task examines the number of reviews for each star rating and compares the total number of positive and negative reviews.

Positive reviews have 4–5 stars, while negative reviews have 1–2 stars. Three-star reviews were removed during Phase 1.


In [ ]:

# Count reviews for each star rating
stars_pd = (df.groupBy("stars")
              .count()
              .orderBy("stars")
              .toPandas())

# Count positive and negative reviews
label_pd = (df.groupBy("label")
              .count()
              .orderBy("label")
              .toPandas())

# Replace numerical labels with names
label_pd["label"] = label_pd["label"].map({
    0: "Negative",
    1: "Positive"
})

print("Reviews per star rating:")
print(stars_pd)

print("\nPositive vs Negative reviews:")
print(label_pd)


In [ ]:

fig, ax = plt.subplots(1, 2, figsize=(11, 4))

# Reviews per star rating
sns.barplot(
    data=stars_pd,
    x="stars",
    y="count",
    ax=ax[0]
)

ax[0].set_title("Reviews per Star Rating")
ax[0].set_xlabel("Star Rating")
ax[0].set_ylabel("Number of Reviews")

# Positive vs Negative reviews
sns.barplot(
    data=label_pd,
    x="label",
    y="count",
    ax=ax[1]
)

ax[1].set_title("Positive vs Negative Reviews")
ax[1].set_xlabel("Sentiment")
ax[1].set_ylabel("Number of Reviews")

plt.tight_layout()

plt.savefig("figures/01_label_distribution.png", dpi=300)

plt.show()



**Interpretation:**

The dataset contains significantly more positive reviews than negative reviews. Positive reviews represent approximately 74.4% of the dataset, while negative reviews represent 25.6%.

This shows that the dataset is imbalanced, with almost three positive reviews for every negative review. Therefore, accuracy alone may not be enough to evaluate the models in Phase 3. Metrics such as Precision, Recall, and F1-score will also be important.



### Task 2 — Review Length

This task compares the length of positive and negative reviews by calculating the average, median, and 95th percentile of the number of words.

A histogram will also be created to visualize the differences in review length between the two sentiment classes.


In [ ]:

# Calculate the number of characters and words in each review
df_len = (df.withColumn("n_chars", length("text"))
            .withColumn("n_words", size(split(col("text"), r"\s+"))))

# Calculate average, median, and 95th percentile
length_stats = (df_len.groupBy("label")
    .agg(
        avg("n_words").alias("avg_words"),
        expr("percentile_approx(n_words, 0.5)").alias("median_words"),
        expr("percentile_approx(n_words, 0.95)").alias("p95_words")
    )
    .orderBy("label")
)

print("Review length statistics:")
length_stats.show()


In [ ]:

# Select review length and sentiment
# Use 1% of the dataset for plotting
len_pd = (df_len.select("n_words", "label")
                .sample(fraction=0.01, seed=42)
                .toPandas())

# Remove the longest 1% of reviews for a clearer chart
len_pd = len_pd[
    len_pd["n_words"] <= len_pd["n_words"].quantile(0.99)
]

# Convert numerical labels into readable names
len_pd["label"] = len_pd["label"].map({
    0: "Negative",
    1: "Positive"
})

print("Visualization sample size:", len(len_pd))
print(len_pd.head())


In [ ]:

plt.figure(figsize=(9, 4))

sns.histplot(
    data=len_pd,
    x="n_words",
    hue="label",
    bins=60,
    stat="density",
    common_norm=False
)

plt.title("Review Length (Words) by Sentiment")
plt.xlabel("Words per Review")
plt.ylabel("Density")

plt.tight_layout()

plt.savefig("figures/02_review_length.png", dpi=300)

plt.show()



**Interpretation:**

The histogram compares the distribution of review lengths for positive and negative reviews.

The average and median values help identify which sentiment tends to have longer reviews. The results can help us understand whether review length is related to customer sentiment.
